Loading dataset

In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter

def squad2_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    rows = []
    for article in squad["data"]:
        title = article["title"]
        for para_i, para in enumerate(article["paragraphs"]):
            context = para["context"]
            paragraph_id = f"{title}__{para_i}"

            for qa in para["qas"]:
                qid = qa["id"]
                question = qa["question"]
                is_impossible = qa.get("is_impossible", False)
                can_answer = "yes" if not is_impossible else "no"

                if can_answer == "yes" and len(qa.get("answers", [])) > 0:
                    answer_text  = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text  = "NO_ANSWER"
                    answer_start = -1

                rows.append({
                    "id":           qid,
                    "title":        title,
                    "paragraph_id": paragraph_id,
                    "context":      context,
                    "question":     question,
                    "can_answer":   can_answer,
                    "answer_text":  answer_text,
                    "answer_start": answer_start
                })

    return pd.DataFrame(rows)


def add_basic_features(df):
    df = df.copy()

    if "can_answer" in df.columns:
        df["y"] = (df["can_answer"].astype(str).str.lower() == "yes").astype(int)
    elif "is_impossible" in df.columns:
        df["y"] = (df["is_impossible"] == False).astype(int)
    else:
        df["y"] = (df["answer_text"] != "NO_ANSWER").astype(int)

    df["q_len_char"] = df["question"].astype(str).str.len()
    df["c_len_char"] = df["context"].astype(str).str.len()
    df["q_len_tok"]  = df["question"].astype(str).str.split().str.len()
    df["c_len_tok"]  = df["context"].astype(str).str.split().str.len()

    if "answer_text" in df.columns:
        df["a_len_char"] = df["answer_text"].astype(str).str.len()
        df["a_len_tok"]  = df["answer_text"].astype(str).str.split().str.len()
        df.loc[df["answer_text"] == "NO_ANSWER", ["a_len_char", "a_len_tok"]] = 0

    return df


train_df = squad2_to_df("train_sampled.json")
val_df   = squad2_to_df("val_sampled.json")
test_df  = squad2_to_df("test_sampled.json")

train_df = add_basic_features(train_df)
val_df   = add_basic_features(val_df)
test_df  = add_basic_features(test_df)

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

Train shape: (8001, 15)
Val shape:   (996, 15)
Test shape:  (1004, 15)


Upload del best model 

In [2]:
import torch
from transformers import DebertaV2ForQuestionAnswering

SAVE_PATH  = "deberta_squad_finetuned"
device = "mps" if torch.backends.mps.is_available() else "cpu"

best_model = DebertaV2ForQuestionAnswering.from_pretrained(SAVE_PATH)
best_model.to(device)
best_model.eval()

print("Best model loaded.")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 10334.23it/s]


Best model loaded.


# Hybrid Retrieval-Augmented Question Answering Pipeline

The presented code implements a Retrieval-Augmented Generation (RAG) pipeline for extractive Question Answering (QA) using a hybrid retrieval strategy. The system combines:

- semantic retrieval based on sentence embeddings

- lexical retrieval based on TF-IDF

- a transformer QA model that extracts the answer span from the retrieved contexts

The pipeline follows a standard architecture used in modern QA systems.

# Hybrid Retrieval-Augmented Question Answering Pipeline

The following code implements a **Retrieval-Augmented Generation (RAG) pipeline** for **extractive Question Answering (QA)**.  
The system combines three main components:

- **Semantic retrieval** using sentence embeddings  
- **Lexical retrieval** using TF-IDF  
- A **transformer QA model** that extracts answer spans from retrieved contexts  

This architecture is widely used in **modern open-domain Question Answering systems**.


## 1. Installing the Embedding Model

The library `sentence-transformers` provides pretrained models that convert text into **dense semantic embeddings**.

An embedding model defines a mapping

$$
f(text) \rightarrow \mathbb{R}^d
$$

where each sentence is represented as a vector in a $d$-dimensional space.

Texts with similar meaning produce **similar vectors**.

Example:

- *What is the capital of France?*  
- *What city is the capital of France?*

Even though the wording differs, their semantic meaning is similar, so their embeddings will be close in the vector space.

This property enables **semantic search**.


## 2. Building the Context Index

The goal of this step is to construct the **retrieval corpus** used by the system.

Two lists are created:

- **all_contexts** → original contexts  
- **all_contexts_with_title** → contexts enriched with titles  

This enriched representation helps the retrieval model capture **additional semantic signals**.

### Removing Duplicate Contexts

Datasets such as **SQuAD** contain multiple questions referring to the **same paragraph**.

To avoid inserting duplicate contexts into the retrieval index, the code keeps track of contexts that have already been processed.

This ensures that **each paragraph is indexed only once**, reducing memory usage and preventing redundant retrieval results.


### Enriching Contexts with Titles

Each context is augmented with its article title.

Example:

Afghanistan. The Taliban regrouped in Pakistan after 2002...

Titles contain **strong semantic information** that helps retrieval.

For example, the question:

*Where did the Taliban regroup after 2002?*

is easier to match with the correct context if the title **Afghanistan** is included.

## 3. Computing Semantic Embeddings

The model used is:

```
multi-qa-MiniLM-L6-cos-v1
```

This model is specifically optimized for **question-answer retrieval tasks**.

Each context $c_i$ is converted into a vector:

$$
c_i \in \mathbb{R}^{384}
$$

All context embeddings form a matrix:

$$
C =
\begin{bmatrix}
c_1 \\
c_2 \\
\vdots \\
c_n
\end{bmatrix}
$$

where:

- $n$ = number of contexts  
- $d$ = embedding dimension.

These embeddings allow efficient **semantic similarity search**.


## 4. TF-IDF Lexical Retrieval

In addition to semantic embeddings, the system constructs a **TF-IDF index**.

TF-IDF represents documents as vectors of **weighted word frequencies**.

The TF-IDF weight of term $j$ in document $i$ is

$$
w_{i,j} = tf_{i,j} \cdot idf_j
$$

where

$$
idf_j = \log\left(\frac{N}{df_j}\right)
$$

and

- $tf_{i,j}$ = frequency of term $j$ in document $i$  
- $df_j$ = number of documents containing term $j$  
- $N$ = total number of documents.

TF-IDF gives higher importance to **rare but informative words**.

Examples include:

- Taliban  
- Eritrea  
- NARA

These terms strongly identify specific contexts.

### Why Using N-grams

The parameter

```
ngram_range = (1,2)
```

means the model includes:

- **unigrams** (single words)
- **bigrams** (two-word expressions)

Examples:

- national archives  
- military operation  

This improves lexical matching between questions and contexts.


## 5. Hybrid Retrieval

The retrieval function combines **semantic similarity** and **lexical similarity**.

The final score is computed as

$$
score = \alpha \cdot s_{semantic} + (1-\alpha)\cdot s_{lexical}
$$

where

- $s_{semantic}$ = cosine similarity between embeddings  
- $s_{lexical}$ = TF-IDF similarity  
- $\alpha$ controls the relative importance of each component.

In this implementation:

```
alpha = 0.7
```

Meaning:

- 70% semantic similarity  
- 30% lexical similarity  

This hybrid approach improves retrieval robustness because:

- semantic search captures **meaning and synonyms**
- lexical search captures **exact keyword matches**


## 6. RAG Question Answering Pipeline

The complete pipeline follows this structure:

```
Question
 ↓
Hybrid Retriever
 ↓
Top-k contexts
 ↓
Transformer QA model
 ↓
Answer span
```

First, the retriever selects the **k most relevant contexts**.

Then the QA model processes each context together with the question.


## 7. Tokenization for Question Answering

The input to the transformer is structured as:

```
[CLS] question [SEP] context [SEP]
```

Only the **context** is truncated if the sequence becomes too long.

This ensures that the **full question is always preserved**.


## 8. Sliding Window for Long Contexts

If the context exceeds the maximum sequence length, it is split into **overlapping segments** using a sliding window.

Example:

```
chunk 1
chunk 2
chunk 3
```

The overlap ensures that answers located near chunk boundaries are **not lost**.


## 9. Model Outputs

The QA model predicts two vectors:

- **start_logits**
- **end_logits**

These represent the probabilities that a token is the **start** or **end** of the answer span.


## 10. Selecting the Answer Span

The score of a candidate span is computed as

$$
score(i,j) = start\_logit_i + end\_logit_j
$$

The span with the highest score is selected as the predicted answer.


## 11. Handling Unanswerable Questions

In **SQuAD 2.0**, the `[CLS]` token represents **no answer**.

If the score of `[CLS]` is higher than any candidate span, the system predicts that the **question cannot be answered from the retrieved contexts**.


## Final Architecture

```
               Question
                    │
                    ▼
         Hybrid Retriever
   (semantic + lexical search)
                    │
                    ▼
             Top-k contexts
                    │
                    ▼
        Transformer QA Model
                    │
                    ▼
               Answer Span
```


## Conclusion

This code implements a complete **Retrieval-Augmented Question Answering system**, consisting of:

1. Context indexing  
2. Hybrid semantic–lexical retrieval  
3. Top-k context selection  
4. Transformer-based answer extraction  
5. Detection of unanswerable questions  

Such architectures represent the **standard approach in modern document-based QA systems** and form the foundation of many real-world **RAG applications**.

In [7]:
# ── 1. Installing the embedding model ──────────────────────────────────

# !pip install sentence-transformers -q
# Sentence Transformers produce dense vector representations (embeddings) for text

# ── 2. Building the context index ────────────
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import pandas as pd

all_contexts = [] # original contexts
all_contexts_with_title = [] # contexts enriched with titles 

# removing duplicated contexts across train/val/test
seen = set() 
for df in [train_df, val_df, test_df]:
    for _, row in df.iterrows():
        ctx   = row["context"]
        title = str(row["title"]).replace("_", " ")
        if ctx not in seen:
            seen.add(ctx)
            all_contexts.append(ctx)
            all_contexts_with_title.append(f"{title}. {ctx}")

print(f"Contexts in the index: {len(all_contexts)}")

# Sentence embeddings with the enriched text (adding title helps the model understand the topic of the context)
print("Loading embedding model...")
embedder = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

print("Computing semantic embeddings...")
ctx_embeddings = embedder.encode(
    all_contexts_with_title,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# TF-IDF on the enriched text
print("Building TF-IDF index...")
vectorizer   = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(all_contexts_with_title)

# ── 3. Hybrid Retriever ───────────────────────────────
def retrieve(question, k=10, alpha=0.7):
    # enriched question
    enriched_q = f"question: {question}"

    q_emb      = embedder.encode([enriched_q], convert_to_numpy=True)
    sem_scores = cosine_similarity(q_emb, ctx_embeddings).flatten()

    q_tfidf    = vectorizer.transform([enriched_q])
    lex_scores = cosine_similarity(q_tfidf, tfidf_matrix).flatten()

    sem_scores = (sem_scores - sem_scores.min()) / (sem_scores.max() - sem_scores.min() + 1e-9)
    lex_scores = (lex_scores - lex_scores.min()) / (lex_scores.max() - lex_scores.min() + 1e-9)

    combined = alpha * sem_scores + (1 - alpha) * lex_scores
    top_k    = combined.argsort()[-k:][::-1]
    return [(all_contexts[i], float(combined[i])) for i in top_k]

# ── 4. Pipeline RAG ───────────────────────────────────
def rag(question, model, tokenizer, device, k=10, alpha=0.7, verbose=True):
    retrieved = retrieve(question, k=k, alpha=alpha)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Question: {question}")
        print(f"{'='*60}")
        print(f"\n Top {k} retrieved contexts:")
        for j, (ctx, score) in enumerate(retrieved):
            print(f"\n  [{j+1}] Score: {score:.4f}")
            print(f"  {ctx[:200]}...")

    best_answer     = None
    best_context    = None
    best_span_score = -float("inf")

    model.eval()
    for ctx, retrieval_score in retrieved:
        inputs = tokenizer(
            question,
            ctx,
            max_length=384,
            truncation="only_second",
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
            return_tensors="pt"
        )

        offset_mapping = inputs.pop("offset_mapping")
        inputs.pop("overflow_to_sample_mapping")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        for i in range(len(outputs.start_logits)):
            start_logits = outputs.start_logits[i]
            end_logits   = outputs.end_logits[i]
            offsets      = offset_mapping[i]

            null_score = start_logits[0] + end_logits[0]
            span_start = int(start_logits[1:].argmax()) + 1
            span_end   = int(end_logits[1:].argmax()) + 1
            span_score = start_logits[span_start] + end_logits[span_end]

            if span_score > null_score and span_score > best_span_score:
                best_span_score = span_score.item()
                if span_end < span_start:
                    span_end = span_start
                start_char   = offsets[span_start][0].item()
                end_char     = offsets[span_end][1].item()
                best_answer  = ctx[start_char:end_char]
                best_context = ctx

    if verbose:
        print(f"\n{'='*60}")
        if best_answer:
            print(f"Answer: {best_answer}")
            print(f"\n From the context:\n{best_context[:300]}...")
        else:
            print(f"No answer found in the retrieved contexts.")
        print(f"{'='*60}\n")

    return best_answer

# ── 5. Test ───────────────────────────────────────────
# National Archives
rag("How did the National Archives make its documents available online?", best_model, tokenizer, device)
rag("How many digital records does NARA store?",                          best_model, tokenizer, device)
rag("Which university is near the second National Archives facility?",    best_model, tokenizer, device)

# Church and State
rag("Why did early immigrants come to America?",                          best_model, tokenizer, device)
rag("Who was persecuting the Puritans in England?",                       best_model, tokenizer, device)

# Eritrea
rag("What European country influenced Eritrean cuisine?",                 best_model, tokenizer, device)
rag("What is the traditional alcoholic drink of Eritrea made from?",      best_model, tokenizer, device)

# Afghanistan
rag("Where did the Taliban regroup after 2002?",                          best_model, tokenizer, device)
rag("What military operation took place in southern Afghanistan in 2010?", best_model, tokenizer, device)

# Charleston
rag("How did Charles Town become wealthy?",                               best_model, tokenizer, device)
rag("When was the American version of the Spoleto festival created?",     best_model, tokenizer, device)

# Questions with no answer
rag("What is the price of Bitcoin today?",                                best_model, tokenizer, device)
rag("Who is the current CEO of Apple?",                                   best_model, tokenizer, device)

Contesti totali nell'indice: 2140
Caricando modello embedding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8540.68it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolando embeddings dei contesti...


Batches: 100%|██████████| 34/34 [00:10<00:00,  3.22it/s]


Costruendo indice TF-IDF...
Indice ibrido pronto.

Domanda: How did the National Archives make its documents available online?

📚 Top 10 contesti recuperati:

  [1] Score: 0.9657
  The first Archivist, R.D.W. Connor, began serving in 1934, when the National Archives was established by Congress. As a result of a first Hoover Commission recommendation, in 1949 the National Archive...

  [2] Score: 0.9478
  In July 2007, the National Archives announced it would make its collection of Universal Newsreels from 1929 to 1967 available for purchase through CreateSpace, an Amazon.com subsidiary. During the ann...

  [3] Score: 0.9333
  The National Archives and Records Administration (NARA) is an independent agency of the United States government charged with preserving and documenting government and historical records and with incr...

  [4] Score: 0.8525
  NARA also maintains the Presidential Library system, a nationwide network of libraries for preserving and making available the documents o

' John Sculley'

It gives as an answer only few word (or just one) because the model is an encoder and retrieves the information from the text, so since it's not generative if it doesn't find a match in the following/previous words, it retrieves only a word.

In [8]:
# Cerca manualmente se il contesto giusto è nel dataset
query = "western Pakistan Taliban regroup"
for ctx in all_contexts:
    if "western Pakistan" in ctx and "Taliban" in ctx:
        print(ctx[:300])
        print()